# CIS6005 Computational Intelligence
## Notebook 12 — Save Best Model (Production Package)
**Student Health Risk Prediction | Kaggle PS S6E7**

---
### Purpose
Package the complete prediction pipeline into one **production bundle** for the Streamlit app.

The bundle contains:
- Champion ML model
- Feature scaler
- Label encoder
- Categorical encoders
- Feature names list
- Performance metadata

In [1]:
# ============================================================
# IMPORTS & SETUP
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import datetime
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report

PROJECT_ROOT = Path.cwd().parent
PROC_DATA    = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR   = PROJECT_ROOT / 'models'

print('Ready to build production bundle')

Ready to build production bundle


## 1. Load All Pipeline Components

In [2]:
# ============================================================
# SECTION 1: Load All Components
# ============================================================

X_val   = np.load(PROC_DATA / 'X_val.npy')
y_val   = np.load(PROC_DATA / 'y_val.npy')

label_encoder        = joblib.load(MODELS_DIR / 'label_encoder.joblib')
scaler               = joblib.load(MODELS_DIR / 'scaler.joblib')
categorical_encoders = joblib.load(MODELS_DIR / 'categorical_encoders.joblib')
feature_names        = joblib.load(MODELS_DIR / 'feature_names.joblib')

if (MODELS_DIR / 'model_tuned_best.joblib').exists():
    champion_model = joblib.load(MODELS_DIR / 'model_tuned_best.joblib')
    champion_name  = 'Tuned Best Model'
else:
    champion_model = joblib.load(MODELS_DIR / 'model_random_forest.joblib')
    champion_name  = 'Random Forest'

CLASS_NAMES = list(label_encoder.classes_)

print(f'Champion  : {champion_name}')
print(f'Features  : {len(feature_names)}')
print(f'Classes   : {CLASS_NAMES}')

Champion  : Tuned Best Model
Features  : 19
Classes   : ['at-risk', 'fit', 'unhealthy']


## 2. Final Performance Report

In [3]:
# ============================================================
# SECTION 2: Final Validation Metrics
# ============================================================

y_pred    = champion_model.predict(X_val)
final_acc = accuracy_score(y_val, y_pred)
final_f1w = f1_score(y_val, y_pred, average='weighted')
final_f1m = f1_score(y_val, y_pred, average='macro')

print('=' * 60)
print('  FINAL MODEL PERFORMANCE REPORT')
print('=' * 60)
print(f'  Model              : {champion_name}')
print(f'  Validation Accuracy: {final_acc:.4f}  ({final_acc*100:.2f}%)')
print(f'  F1 (Weighted)      : {final_f1w:.4f}')
print(f'  F1 (Macro)         : {final_f1m:.4f}')
print('=' * 60)
print()
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES))

  FINAL MODEL PERFORMANCE REPORT
  Model              : Tuned Best Model
  Validation Accuracy: 0.9991  (99.91%)
  F1 (Weighted)      : 0.9991
  F1 (Macro)         : 0.9972

              precision    recall  f1-score   support

     at-risk       1.00      1.00      1.00    118512
         fit       0.99      1.00      1.00      7961
   unhealthy       0.99      1.00      1.00     11545

    accuracy                           1.00    138018
   macro avg       1.00      1.00      1.00    138018
weighted avg       1.00      1.00      1.00    138018



## 3. Save Production Bundle

In [4]:
# ============================================================
# SECTION 3: Package and Save Production Bundle
# ============================================================

production_bundle = {
    'model'               : champion_model,
    'model_name'          : champion_name,
    'model_type'          : type(champion_model).__name__,
    'scaler'              : scaler,
    'label_encoder'       : label_encoder,
    'categorical_encoders': categorical_encoders,
    'feature_names'       : feature_names,
    'class_names'         : CLASS_NAMES,
    'class_mapping'       : dict(zip(
                                label_encoder.classes_,
                                label_encoder.transform(label_encoder.classes_)
                            )),
    'val_accuracy'        : round(final_acc, 4),
    'val_f1_weighted'     : round(final_f1w, 4),
    'val_f1_macro'        : round(final_f1m, 4),
    'project'             : 'CIS6005 Student Health Risk Prediction',
    'competition'         : 'Kaggle Playground Series S6E7',
    'created_at'          : datetime.datetime.now().isoformat()
}

joblib.dump(production_bundle, MODELS_DIR / 'production_bundle.joblib')
joblib.dump(champion_model,    MODELS_DIR / 'best_model_final.joblib')

print('=' * 60)
print('  PHASE 12 COMPLETE — Production Model Saved')
print('=' * 60)
print(f'  Model           : {champion_name}')
print(f'  Accuracy        : {final_acc*100:.2f}%')
print(f'  F1 (Weighted)   : {final_f1w:.4f}')
print('=' * 60)
print('  Saved files:')
print('  models/production_bundle.joblib  (Full pipeline)')
print('  models/best_model_final.joblib   (Model only)')
print('=' * 60)
print('  Ready for Phase 13: Streamlit Web Application')
print('=' * 60)

  PHASE 12 COMPLETE — Production Model Saved
  Model           : Tuned Best Model
  Accuracy        : 99.91%
  F1 (Weighted)   : 0.9991
  Saved files:
  models/production_bundle.joblib  (Full pipeline)
  models/best_model_final.joblib   (Model only)
  Ready for Phase 13: Streamlit Web Application
